### [ ANN 모델 구성 + 수동(Manual) 역전파 ]
```text
- ANN : 입력층 + 은닉층 + 출력층
    * 입력층 
        - 데이터를 받아서 그대로 다음 층으로 전달하는 역할
        - 가중치(w), 절편(b) : 없음
        - 활성화함수         : 미사용
        - 연산              : 하지 않음.
        
    * 은닉층
        - 입력 × 가중치 + 편향 → 활성화함수 적용
        - 가중치(w), 절편(b) : 있음
        - 활성화함수         : 사용
        - 연산              : 수행함

    * 출력층
        - 마지막 은닉층 결과 × 가중치 + 편향 → (필요시 활성화함수)
        - 가중치(w), 절편(b) : 있음
        - 활성화함수         : 필요 시 적용 (회귀 → 미사용)
        - 연산              : 수행함
``` 

**이번 노트북의 목적**
- 1번 노트북(`ex01_ann_model.ipynb`)은 순전파(forward) + 손실 계산까지만 진행 → 가중치가 업데이트되지 않아 '학습'이 아니었음
- 이번에는 `loss.backward()` 같은 **autograd를 전혀 사용하지 않고**, 미분 공식을 직접 코드로 구현해서 역전파(backward) + 가중치 갱신까지 수행함

**모델 구조**
```text
X(4) → Linear(W1,b1) → Z1 → ReLU → A1 → Linear(W2,b2) → Z2(예측값)
```

**순전파 수식**
- Z1 = X · W1^T + b1
- A1 = ReLU(Z1)
- Z2 = A1 · W2^T + b2   (회귀 → 출력층 활성화함수 없음)
- L  = mean( (Z2 - y)^2 )   (MSE)

**역전파 수식 (연쇄법칙을 손으로 전개)**
```text
dL/dZ2 = 2*(Z2 - y) / N
dL/dW2 = dZ2.T @ A1          dL/db2 = sum(dZ2, dim=0)
dL/dA1 = dZ2 @ W2
dL/dZ1 = dA1 * ReLU'(Z1)      (Z1>0 이면 1, 아니면 0)
dL/dW1 = dZ1.T @ X            dL/db1 = sum(dZ1, dim=0)
```

In [1]:
## ------------------------------------
## 모듈 로딩
## ------------------------------------
import torch 
import torch.nn as nn

torch.manual_seed(1)

In [2]:
## ------------------------------------
## 데이터와 정답 텐서 준비 (ex01과 동일)
## ------------------------------------
featureTN = torch.tensor([[11,22,33,44],
                          [22,33,44,55],
                          [33,44,55,66],
                          [44,55,66,77],
                          [55,66,77,88]], dtype=torch.float32)

targetTN = torch.tensor([55, 66, 77, 88, 99], dtype=torch.float32).reshape(-1, 1)

print("피쳐/데이터 :", featureTN.shape, "타겟/정답 :", targetTN.shape)

피쳐/데이터 : torch.Size([5, 4]) 타겟/정답 : torch.Size([5, 1])


In [3]:
## -------------------------------------------------------------
## Sequential로 구조만 확인 (초기 가중치 생성용) -> 이후 순수 텐서로 분리해서 사용
##            입력       출력(퍼셉트론/노드 수)        활성함수   
## 입력층 :    피쳐 4                4
## 은닉층 :     4                   10                 ReLU
## 출력층 :     10                   1                 없음(회귀)
## -------------------------------------------------------------
model1 = nn.Sequential(
    nn.Linear(4, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
)

display(model1)

Sequential(
  (0): Linear(in_features=4, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=1, bias=True)
)

In [4]:
## -------------------------------------------------------------
## 파라미터를 순수 텐서(autograd 미사용)로 분리
## -> .data 로 꺼내면 requires_grad=False 인 일반 텐서가 됨
## -------------------------------------------------------------
W1 = model1[0].weight.data.clone()   # (10, 4)
b1 = model1[0].bias.data.clone()     # (10,)
W2 = model1[2].weight.data.clone()   # (1, 10)
b2 = model1[2].bias.data.clone()     # (1,)

print("W1", W1.shape, " b1", b1.shape)
print("W2", W2.shape, " b2", b2.shape)

W1 torch.Size([10, 4])  b1 torch.Size([10])
W2 torch.Size([1, 10])  b2 torch.Size([1])


In [6]:
## -------------------------------------------------------------
## 활성화함수 + 미분 함수 직접 구현
## -------------------------------------------------------------
def relu(z):
    return torch.clamp(z, min=0)

def relu_deriv(z):
    # ReLU'(z) = 1 (z>0), 0 (z<=0)
    return (z > 0).float()


## -------------------------------------------------------------
## 순전파(forward) 직접 구현 (autograd 미사용 - 그냥 텐서 연산)
## -------------------------------------------------------------
def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1.T + b1     # (N,4)@(4,10) -> (N,10)
    A1 = relu(Z1)          # (N,10)
    Z2 = A1 @ W2.T + b2    # (N,10)@(10,1) -> (N,1)  <- 예측값(pre_y)
    return Z1, A1, Z2


## -------------------------------------------------------------
## 손실함수(MSE) 직접 구현
## -------------------------------------------------------------
def mse_loss(pred, target):
    return ((pred - target) ** 2).mean()

In [7]:
## -------------------------------------------------------------
## 역전파(backward) 직접 구현 - loss.backward() 사용 안 함!
## 연쇄법칙(chain rule)을 손으로 풀어서 그대로 코드로 옮긴 것
## -------------------------------------------------------------
def backward(X, y, Z1, A1, Z2, W2):
    N = X.shape[0]

    ## [출력층] dL/dZ2 : MSE를 Z2로 미분
    dZ2 = 2 * (Z2 - y) / N            # (N,1)

    ## [출력층] 가중치/절편 기울기
    dW2 = dZ2.T @ A1                  # (1,N)@(N,10) -> (1,10)
    db2 = dZ2.sum(dim=0)              # (1,)

    ## [은닉층] dL/dA1 : 출력층 가중치를 거슬러 전달
    dA1 = dZ2 @ W2                    # (N,1)@(1,10) -> (N,10)

    ## [은닉층] dL/dZ1 : ReLU 미분 곱하기 (활성화함수 통과 전으로 되돌리기)
    dZ1 = dA1 * relu_deriv(Z1)        # (N,10)

    ## [은닉층] 가중치/절편 기울기
    dW1 = dZ1.T @ X                   # (10,N)@(N,4) -> (10,4)
    db1 = dZ1.sum(dim=0)              # (10,)

    return dW1, db1, dW2, db2

In [8]:
## ------------------------------------
## 학습 전 예측값 확인 (아직 가중치 갱신 안 함)
## ------------------------------------
_, _, pre_y_before = forward(featureTN, W1, b1, W2, b2)
print("[학습 전] pre_y =>", pre_y_before)
print("[학습 전] loss  =>", mse_loss(pre_y_before, targetTN).item())

[학습 전] pre_y => tensor([[-1.4876],
        [-0.1564],
        [ 1.1964],
        [ 2.5492],
        [ 3.9020]])
[학습 전] loss  => 5931.8359375


In [9]:
## -------------------------------------------------------------
## 학습 루프 : 순전파 -> 손실계산 -> (수동)역전파 -> 가중치 갱신 반복
## * 데이터 값 자체가 크기 때문에(11~88) 학습률을 작게 잡음
##   (스케일링을 안 했을 때 학습률을 왜 작게 잡아야 하는지 보여주는 좋은 예시)
## -------------------------------------------------------------
lr = 1e-6
epochs = 2000

for epoch in range(1, epochs + 1):
    ## 1) 순전파
    Z1, A1, Z2 = forward(featureTN, W1, b1, W2, b2)

    ## 2) 손실 계산
    loss = mse_loss(Z2, targetTN)

    ## 3) 역전파 (직접 구현한 미분 공식 사용, autograd 미사용)
    dW1, db1, dW2, db2 = backward(featureTN, targetTN, Z1, A1, Z2, W2)

    ## 4) 가중치 갱신 (경사하강법)
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 200 == 0 or epoch == 1:
        print(f"epoch {epoch:4d}  loss {loss.item():.4f}")

epoch    1  loss 5931.8359
epoch  200  loss 46.1394
epoch  400  loss 43.1521
epoch  600  loss 40.5032
epoch  800  loss 38.3388
epoch 1000  loss 36.2995
epoch 1200  loss 34.3744
epoch 1400  loss 32.5542
epoch 1600  loss 30.8306
epoch 1800  loss 29.1965
epoch 2000  loss 27.6455


In [11]:
## ------------------------------------
## 학습 후 예측값과 정답 비교
## ------------------------------------
_, _, pre_y_after = forward(featureTN, W1, b1, W2, b2)

print(f"[학습 전] pre_y => {pre_y_before}")
print(f"[학습 후] pre_y => {pre_y_after}")
print(f"y (정답)        => {targetTN}")

[학습 전] pre_y => tensor([[-1.4876],
        [-0.1564],
        [ 1.1964],
        [ 2.5492],
        [ 3.9020]])
[학습 후] pre_y => tensor([[ 46.4074],
        [ 61.0165],
        [ 75.5885],
        [ 90.1605],
        [104.7325]])
y (정답)        => tensor([[55.],
        [66.],
        [77.],
        [88.],
        [99.]])


### 정리
```text
- ex01 : 순전파 + 손실계산만 -> 가중치 안 바뀜 -> 학습 아님
- ex02 (이 노트북) : 순전파 + 손실계산 + '직접 구현한' 역전파 + 가중치 갱신을 반복 -> 진짜 학습

- autograd(loss.backward())를 쓰지 않고도 학습이 되는 이유
  : PyTorch의 backward()가 내부적으로 하는 일이
    바로 이 노트북에서 손으로 구현한 '연쇄법칙에 따른 기울기 계산'과 동일하기 때문

- 학습률(lr)이 매우 작은 이유
  : 입력 데이터가 스케일링(정규화)되지 않고 11~88 범위 그대로라서
    기울기 값도 커짐 -> lr이 크면 가중치가 발산(loss가 폭주)할 수 있음
  : 실전에서는 StandardScaler 등으로 스케일링 후 학습률을 좀 더 크게 잡는 것이 일반적
```